# A Parser for Regular Expression

## Type Checking

The functions in this notebook carry *type annotations*.  In OCaml, these annotations are checked by the
compiler before a cell is executed, so there is no need for a separate type checker like *basedpyright*.
A cell that contains a type error is not executed at all.

This notebook implements a parser for regular expressions. The parser that is implemented in the function `parse` parses a regular expression
according to the following <em style="color:blue">EBNF grammar</em>.
```
   regExp  → product ('+' product)*
   product → factor factor*
   factor  → atom '*'*
   atom    → '(' regExp ')' | LETTER | '𝜀' | '∅'
```
The parse tree is represented as a value of the type `regexp`:
- a letter `c` is represented as `Char c`,
- The character `'∅'` is interpreted as the regular expression $\emptyset$ denoting the empty set and is represented as `Empty`,
- The character `'𝜀'` represents the regular expression $\varepsilon$ denoting the empty string and is represented as `Epsilon`,
- $r_1 r_2$ is represented as `Product (`$r_1$`,` $r_2$`)`,
- $r_1 + r_2$ is represented as `Sum (`$r_1$`,` $r_2$`)`,
- $r^*$ is represented as `Star` $r$.

The parser is implemented as a recursive *top-down* parser.

We start with a definition of the type of the parse trees that are generated.  This type is a *recursive
type*.  Since it is used by several notebooks, it is defined in the notebook `Types.ipynb`, which is loaded by
the function `run_notebook`.  The directive `#show_type` shows its definition.

In [ ]:
#use "../lib/nb.ml";;
run_notebook "Types.ipynb"

In [ ]:
#show_type regexp

In order to tokenize strings, we need regular expressions.  We use the module `Re` from the file
`../lib/regex.ml`, which offers the functions of Python's module `re` on top of the library PCRE2.

In [ ]:
load_lib "regex.ml"

The function `tokenize s` partitions the string `s` into a list of tokens.
It recognizes
- the operator symbols `+` and `*`,
- the parentheses `(`, `)`,
- single upper or lower case letters and single digits,
- the symbol `∅` that matches the empty language,
- the symbol `𝜀` that matches the empty string.

All whitespace characters (and, indeed, all characters that could not be matched) are discarded.
The flag `VERBOSE` allows us to use white space and comments inside the regular expression.

**Function `tokenize s`**
- *Input:* `s` is a string that represents a regular expression.
- *Output:* The list of the tokens of `s`.

In [ ]:
let tokenize (s : string) : string list =
  let regExp = {|
              [+*()]      |  # operators and parentheses
              [a-zA-Z0-9] |  # single characters or digits from the alphabet
              ∅           |  # empty regular expression
              𝜀              # epsilon
              |}
  in
  Re.findall ~flags:Re.[VERBOSE] regExp s

In [ ]:
tokenize "a*bc + ba*c + (𝜀+c*) + ∅*"

The function `parse` takes a string `s` and tries to parse it as a regular expression.
It returns the parse tree.

The functions `parse`, `parseRegExp`, `parseProduct`, `parseFactor`, and `parseAtom` call each other.  For
example, `parse` calls `parseRegExp`, which is only defined below.  In OCaml, a function can only use
functions that have already been defined, unless all of these functions are defined together using
`let rec ... and ...`.  In order to keep every function in a cell of its own, we use *forward references*:
the next cell creates references, i.e. mutable boxes, that will later hold the functions `parseRegExp`,
`parseProduct`, `parseFactor`, and `parseAtom`.  `!parseRegExp_ref` is the function that is currently stored in
the box.  After a function has been defined, it is stored in its box via the operator `:=`.

In [ ]:
type parser = string list -> regexp * string list

let parseRegExp_ref  : parser ref = ref not_yet_defined
let parseProduct_ref : parser ref = ref not_yet_defined
let parseFactor_ref  : parser ref = ref not_yet_defined
let parseAtom_ref    : parser ref = ref not_yet_defined

let show_tokens (tl : string list) : string =
  "[" ^ String.concat ", " (List.map repr tl) ^ "]"

**Function `parse s`**
- *Input:* `s` is a string that represents a regular expression.
- *Output:* The parse tree of `s`.

In [ ]:
let parse (s : string) : regexp =
  let tokenList = tokenize s in
  match !parseRegExp_ref tokenList with
  | regExp, [] -> regExp
  | _, _ -> failwith (Printf.sprintf "Parse Error: could not parse %s" (show_tokens tokenList))

The function `parseRegExp` takes a token list `tokenList` and tries to interpret this list as a regular expression.
It returns the regular expression in the form of a value of type `regexp` and a list of those tokens that could not be parsed.
It is implemented as a <em style="color:blue">top-down-parser.</em>  The `while` loop of the Python version is
replaced by the recursive function `loop`.

The function `parseRegExp` implements the following grammar rule:
```
regExp → product ('+' product)*
```

**Function `parseRegExp tokenList`**
- *Input:* `tokenList` is a list of tokens.
- *Output:* A pair consisting of the parse tree of the regular expression at the beginning of `tokenList` and the list of the remaining tokens.

In [ ]:
let parseRegExp (tokenList : string list) : regexp * string list =
  let rec loop result rest =
    match rest with
    | "+" :: (_ :: _ as rest) ->
        let arg, rest = !parseProduct_ref rest in
        loop (Sum (result, arg)) rest
    | _ -> result, rest
  in
  let result, rest = !parseProduct_ref tokenList in
  loop result rest
;;
parseRegExp_ref := parseRegExp

The function `parseProduct` has to check whether the next token can start an atom.  To this end, we need the
strings of all digits and of all letters.  In Python, these are found in the module `string`.  In OCaml, we
define them ourselves.

In [ ]:
let digits = "0123456789"
let ascii_letters = "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ"

In [ ]:
digits

In [ ]:
ascii_letters

The function `parseProduct` implements the following grammar rule:
```
product → factor factor*
```
A product continues as long as the next token can start an atom, i.e. if it is a letter, a digit, an opening
parenthesis, `∅`, or `𝜀`.  This is checked by the function `starts_atom`.

**Function `parseProduct tokenList`**
- *Input:* `tokenList` is a list of tokens.
- *Output:* A pair consisting of the parse tree of the product at the beginning of `tokenList` and the list of the remaining tokens.

In [ ]:
let starts_atom (t : string) : bool =
  t = "∅" || t = "𝜀" || (String.length t = 1 && String.contains (ascii_letters ^ digits ^ "(") t.[0])

let parseProduct (tokenList : string list) : regexp * string list =
  let rec loop result rest =
    match rest with
    | t :: _ when starts_atom t ->
        let arg, rest = !parseFactor_ref rest in
        loop (Product (result, arg)) rest
    | _ -> result, rest
  in
  let result, rest = !parseFactor_ref tokenList in
  loop result rest
;;
parseProduct_ref := parseProduct

The function `parseFactor` implements the following grammar rule:
```
factor → atom '*'*
```

**Function `parseFactor tokenList`**
- *Input:* `tokenList` is a list of tokens.
- *Output:* A pair consisting of the parse tree of the factor at the beginning of `tokenList` and the list of the remaining tokens.

In [ ]:
let parseFactor (tokenList : string list) : regexp * string list =
  let rec loop atom rest =
    match rest with
    | "*" :: rest -> loop (Star atom) rest
    | _ -> atom, rest
  in
  let atom, rest = !parseAtom_ref tokenList in
  loop atom rest
;;
parseFactor_ref := parseFactor

The function `parseAtom` implements the following grammar rule:
```
atom  → '(' expr ')'
      | '∅'            # denotes empty set ∅
      | '𝜀'            # denotes empty string 𝜀
      | LETTER         # any letter denotes itself
```
Instead of looking at `TokenList[0]` and `TokenList[1:]`, the OCaml function uses *pattern matching* on lists: the
pattern `"(" :: rest` matches every list whose first element is `"("` and binds `rest` to the remaining elements.

**Function `parseAtom tokenList`**
- *Input:* `tokenList` is a list of tokens.
- *Output:* A pair consisting of the parse tree of the atom at the beginning of `tokenList` and the list of the remaining tokens.

In [ ]:
let parseAtom (tokenList : string list) : regexp * string list =
  match tokenList with
  | "(" :: rest -> (
      match parseRegExp rest with
      | regExp, ")" :: rest -> regExp, rest
      | _ -> failwith "Parse Error")
  | "∅" :: rest -> Empty, rest
  | "𝜀" :: rest -> Epsilon, rest
  | s :: rest when String.length s = 1 && String.contains (ascii_letters ^ digits) s.[0] -> Char s.[0], rest
  | _ -> failwith (Printf.sprintf "parse error: %s" (show_tokens tokenList))
;;
parseAtom_ref := parseAtom

In [ ]:
parse "a*"

In [ ]:
parse "1*"

In [ ]:
parse "a**"

In [ ]:
parse "ab"

In [ ]:
parse "abc"

In [ ]:
parse "a+b+c"

In [ ]:
parse "ab+cd*"

In [ ]:
parse "(a+b*)*"

In [ ]:
parse "(a**+b*)*"

In [ ]:
parse "0*"

**Type safety:** In the Python version, `parse('0*')` returns `('0', '*')`, where the string `'0'` is the
character `0`, while the number `0` represents the regular expression $\emptyset$.  These two are easily confused.  In
OCaml, the result is `Star (Char '0')`, while $\emptyset$ is represented by `Empty`.